<img src='https://unlearning-challenge.github.io/Unlearning-logo.png' width='100px'>

# NeurIPS 2023 Machine Unlearning Challenge Starting Kit

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/unlearning-challenge/starting-kit/blob/main/unlearning-CIFAR10.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/unlearning-challenge/starting-kit/main/unlearning-CIFAR10.ipynb)


This notebook is part of the starting kit for the [NeurIPS 2023 Machine Unlearning Challenge](https://unlearning-challenge.github.io/). This notebook explains the pipeline of the challenge and contains sample unlearning and evaluation code.


This notebook has 3 sections:

  * 💾 In the first section we'll load a sample dataset (CIFAR10) and pre-trained model (ResNet18).

  * 🎯 In the second section we'll develop the unlearning algorithm. We start by splitting the original training set into a retain set and a forget set. The goal of an unlearning algorithm is to update the pre-trained model so that it approximates as much as possible a model that has been trained on the retain set but not on the forget set. We provide a simple unlearning algorithm as a starting point for participants to develop their own unlearning algorithms.

  * 🏅 In the third section we'll score our unlearning algorithm using a simple membership inference attacks (MIA). Note that this is a different evaluation than the one that will be used in the competition's submission.
  

We emphasize that this notebook is provided for convenience to help participants quickly get started. Submissions will be scored using a different method than the one provided in this notebook on a different (private) dataset of human faces. To run the notebook, the requirement is to have installed an up-to-date version of Python and Pytorch.

In [7]:
import os
import requests
import numpy as np
#import matplotlib.pyplot as plt
from sklearn import linear_model, model_selection

import torch
from torch import nn
from torch import optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms
from torchvision.utils import make_grid
from torchvision.models import resnet18

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device:", DEVICE.upper())

# manual random seed is used for dataset partitioning
# to ensure reproducible results across runs
RNG = torch.Generator().manual_seed(42)

Running on device: CUDA


# 💾 Download dataset and pre-trained model

In this section, we'll load a sample dataset (CIFAR-10), a pre-trained model (ResNet18) trained on CIFAR-10, plot some images and compute the accuracy of the model on the test set.

In [8]:
# download and pre-process CIFAR10
normalize = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)

train_set = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=normalize
)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)

# we split held out data into test and validation set
held_out = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=normalize
)
# test_set, val_set = torch.utils.data.random_split(held_out, [0.5, 0.5], generator=RNG)

n = len(held_out)            # 10000 images in CIFAR10 test set
n_test = n // 2              # 5000
n_val = n - n_test           # 5000 (use remainder to avoid off-by-one)

test_set, val_set = torch.utils.data.random_split(
    held_out, [n_test, n_val], generator=RNG
)


test_loader = DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2)
val_loader = DataLoader(val_set, batch_size=128, shuffle=False, num_workers=2)

# download the forget and retain index split
local_path = "forget_idx.npy"
if not os.path.exists(local_path):
    response = requests.get(
        "https://storage.googleapis.com/unlearning-challenge/" + local_path
    )
    open(local_path, "wb").write(response.content)
forget_idx = np.load(local_path)

# construct indices of retain from those of the forget set
forget_mask = np.zeros(len(train_set.targets), dtype=bool)
forget_mask[forget_idx] = True
retain_idx = np.arange(forget_mask.size)[~forget_mask]

# split train set into a forget and a retain set
forget_set = torch.utils.data.Subset(train_set, forget_idx)
retain_set = torch.utils.data.Subset(train_set, retain_idx)

forget_loader = torch.utils.data.DataLoader(
    forget_set, batch_size=128, shuffle=True, num_workers=2
)
retain_loader = torch.utils.data.DataLoader(
    retain_set, batch_size=128, shuffle=True, num_workers=2, generator=RNG
)

100%|██████████| 170M/170M [00:13<00:00, 12.5MB/s]


We'll now download the weights of the model trained in CIFAR-10 and load them in a Pytorch model. This model has been trained using SGD with a learning rate of 0.1, momentum of 0.9 and weight decay of 5e-4. It was also trained using data augmentation. In particular, the transforms used to the data were:

```python
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
```


In [9]:
# download pre-trained weights
local_path = "weights_resnet18_cifar10.pth"
if not os.path.exists(local_path):
    response = requests.get(
        "https://storage.googleapis.com/unlearning-challenge/weights_resnet18_cifar10.pth"
    )
    open(local_path, "wb").write(response.content)

weights_pretrained = torch.load(local_path, map_location=DEVICE)

# load model with pre-trained weights
model = resnet18(weights=None, num_classes=10)
model.load_state_dict(weights_pretrained)
model.to(DEVICE)
model.eval();

In [10]:
# download weights of a model trained exclusively on the retain set
local_path = "retrain_weights_resnet18_cifar10.pth"
if not os.path.exists(local_path):
    response = requests.get(
        "https://storage.googleapis.com/unlearning-challenge/" + local_path
    )
    open(local_path, "wb").write(response.content)

weights_pretrained = torch.load(local_path, map_location=DEVICE)

# load model with pre-trained weights
rt_model = resnet18(weights=None, num_classes=10)
rt_model.load_state_dict(weights_pretrained)
rt_model.to(DEVICE)
rt_model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
import sys

import rmu
import sim_npo
import original_method
import original_metrics
import kl_div
import undial
import truth_ratio

import copy

In [42]:
import pandas as pd
import copy

full_model = copy.deepcopy(model)

# --- Unlearn models ---
sim_npo_model = sim_npo.train_simnpo(
    copy.deepcopy(full_model), retain_loader, forget_loader, val_loader,
    alpha=0.9, beta=4.0, delta=0.2
)
undial_model = undial.train_undial(
    copy.deepcopy(full_model), forget_loader, retain_loader
)
rmu_model = rmu.train_rmu(
    copy.deepcopy(full_model), forget_loader, retain_loader
)

models = [
    ("SimNPO", sim_npo_model),
    ("UNDIAL", undial_model),
    ("RMU",    rmu_model),
]


[UNDIAL] ep 1/12 loss_f=9.601 loss_r=9.451 acc_f=0.644 acc_r=0.764
[UNDIAL] ep 2/12 loss_f=7.223 loss_r=10.530 acc_f=0.583 acc_r=0.669
[UNDIAL] ep 3/12 loss_f=2.974 loss_r=4.346 acc_f=0.651 acc_r=0.785
[UNDIAL] ep 4/12 loss_f=2.518 loss_r=3.682 acc_f=0.622 acc_r=0.773
[UNDIAL] ep 5/12 loss_f=2.343 loss_r=3.349 acc_f=0.657 acc_r=0.820
[UNDIAL] ep 6/12 loss_f=2.146 loss_r=2.552 acc_f=0.660 acc_r=0.859
[UNDIAL] ep 7/12 loss_f=1.816 loss_r=2.310 acc_f=0.643 acc_r=0.843
[UNDIAL] ep 8/12 loss_f=1.870 loss_r=1.859 acc_f=0.669 acc_r=0.896
[UNDIAL] ep 9/12 loss_f=1.516 loss_r=1.533 acc_f=0.662 acc_r=0.914
[UNDIAL] ep 10/12 loss_f=1.433 loss_r=1.408 acc_f=0.654 acc_r=0.921
[UNDIAL] ep 11/12 loss_f=1.466 loss_r=1.140 acc_f=0.658 acc_r=0.927
[UNDIAL] ep 12/12 loss_f=1.466 loss_r=1.056 acc_f=0.650 acc_r=0.930
UNDIAL training complete — returning unlearned student model.
[Epoch 1/4] layer='layer3.0.conv1' range=[10,14] steps=352 alpha=1000.00 c=4.0000 k=0.75


[Epoch 2/4] layer='layer4.0.conv1' range=[15,20] steps=352 alpha=1000.00 c=4.0000 k=1.0


[Epoch 3/4] layer='fc' range=[15,20] steps=352 alpha=1000.00 c=4.0000 k=1.0


[Epoch 4/4] layer='layer4.0.downsample.0' range=[15,20] steps=352 alpha=1000.00 c=4.0000 k=1.0


--- RMU Training Complete ---


In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from copy import deepcopy
from itertools import product
from sklearn import linear_model, model_selection

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
criterion = nn.CrossEntropyLoss()


def flat_params(m):
    return torch.cat([p.detach().reshape(-1) for p in m.parameters() if p.requires_grad])


def per_sample_grad(model, x, y, loss_fn):
    """
    Per-sample gradient wrt *base model* parameters.
    x, y are single-sample batches (shape [1, ...]).
    """
    model.zero_grad(set_to_none=True)
    out = model(x)
    loss = loss_fn(out, y)
    grads = torch.autograd.grad(
        loss,
        [p for p in model.parameters() if p.requires_grad],
        retain_graph=False,
        create_graph=False,
    )
    return torch.cat([g.reshape(-1) for g in grads])


def _bn_eval_copy(model: nn.Module) -> nn.Module:
    """
    Make a copy of `model` where all BatchNorm layers are forced into eval mode.
    We use this copy ONLY for influence computation (so BN never sees batch_size=1 in training mode).
    """
    m = deepcopy(model).to(DEVICE)
    m.eval()
    for mod in m.modules():
        if isinstance(mod, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            mod.eval()
    return m


def compute_leakage(unlearned_model, base_model, retrain_model, forget_loader,
                    eps=1e-6, floor_pct=0.01):
    """
    Influence-based leakage:

      delta_true = θ_rt  - θ_base
      delta_unl  = θ_unl - θ_base

      influence_true(i) = g_i^T delta_true
      influence_unl(i)  = g_i^T delta_unl

      leakage_i = |influence_unl(i)| / |influence_true(i)|

    Returns a dict with norms and filtered leakage stats.
    """
    base_for_if = _bn_eval_copy(base_model)

    unlearned_model = unlearned_model.to(DEVICE).eval()
    retrain_model   = retrain_model.to(DEVICE).eval()

    with torch.no_grad():
        delta_true = (flat_params(retrain_model)   - flat_params(base_for_if)).to(DEVICE)
        delta_unl  = (flat_params(unlearned_model) - flat_params(base_for_if)).to(DEVICE)

    infl_true_list = []
    leakage_list   = []

    for xb, yb in forget_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        for i in range(xb.size(0)):
            xi = xb[i:i+1]
            yi = yb[i:i+1]

            g_i = per_sample_grad(base_for_if, xi, yi, criterion)

            infl_true = torch.dot(g_i, delta_true)
            infl_unl  = torch.dot(g_i, delta_unl)

            infl_true_list.append(infl_true.detach().cpu())

            if abs(float(infl_true)) < eps:
                leak = 0.0
            else:
                leak = abs(float(infl_unl)) / (abs(float(infl_true)) + eps)
            leakage_list.append(leak)

    infl_true_tensor = torch.stack(infl_true_list)
    leakage_tensor   = torch.tensor(leakage_list)

    floor = floor_pct * infl_true_tensor.abs().max()
    mask = infl_true_tensor.abs() > floor
    filtered_leak = leakage_tensor[mask]

    return {
        "delta_true_norm": float(delta_true.norm()),
        "delta_unl_norm":  float(delta_unl.norm()),
        "filtered_mean_leakage": float(filtered_leak.mean()) if filtered_leak.numel() > 0 else None,
        "filtered_max_leakage":  float(filtered_leak.max())  if filtered_leak.numel() > 0 else None,
        "filtered_count":        int(mask.sum().item()),
    }

In [44]:
results = []


def eval_metrics(mod, base_model, retrain_model):
    # Accuracy, forget and test sets
    acc_test   = original_metrics.accuracy(mod, test_loader)
    acc_forget = original_metrics.accuracy(mod, forget_loader)

    # MIA Score
    mia_score = float(np.mean(original_metrics.run_simple_mia(mod, forget_loader, test_loader)))

    # KL-Divergence
    kl_forget_vs_rt = kl_div.kl_student_vs_ref(mod, retrain_model, forget_loader, T=4.0)
    kl_test_vs_rt   = kl_div.kl_student_vs_ref(mod, retrain_model, test_loader,   T=4.0)

    # Influence Fucntions
    influence_fucntion_dict = compute_leakage(mod, base_model, retrain_model, forget_loader)

    # Truth ratio
    truth_ratio_forget = truth_ratio.truth_ratio(mod, forget_loader)
    truth_ratio_test   = truth_ratio.truth_ratio(mod, test_loader)
    return {
        "acc_test": acc_test,
        "acc_forget": acc_forget,
        "mia_score": mia_score,
        "kl_forget_vs_rt": kl_forget_vs_rt,
        "kl_test_vs_rt": kl_test_vs_rt,
        "truth_ratio_forget": truth_ratio_forget,
        "truth_ratio_test": truth_ratio_test
    } | influence_fucntion_dict

for name, mod in models:
    metrics = eval_metrics(mod, model, rt_model)
    metrics["method"] = name
    results.append(metrics)

df = pd.DataFrame(results)
df

,acc_test,acc_forget,mia_score,kl_forget_vs_rt,kl_test_vs_rt,truth_ratio_forget,truth_ratio_test,delta_true_norm,delta_unl_norm,filtered_mean_leakage,filtered_max_leakage,filtered_count,method
0,0.8324,0.2324,0.7749,5.654000,1.932520,0.289821,0.804711,40.817757,17.487980,0.447367,13.741479,458,SimNPO
1,0.8302,0.6496,0.6128,6.600956,5.487359,0.431347,0.575130,40.817757,12.691652,0.378715,5.157058,458,UNDIAL
2,0.8690,0.9754,0.5603,1.721942,1.454266,0.967471,0.863168,40.817757,6.408556,0.184715,3.929513,458,RMU
